# Phase B2: Per-Seed Ranking Stability
## ThermoRG — HBO Two-Stage Heuristic Validation

**Goal:** Test whether J_topo-based ranking is stable across training seeds.

**Motivation (Reviewer B2):** The paper claims J_topo ranking has "high per-seed variance" justifying the two-stage approach (D≥48 → J_topo ranking). This experiment provides empirical backing.

**Hypothesis:**
- If |Spearman ρ| < 0.7: J_topo ranking is seed-dependent → two-stage heuristic is empirically justified
- If ρ ≤ -0.7: J_topo ranking is reliable across seeds → two-stage may be unnecessary

**Setup:** 5 architectures × 5 seeds × 10 epochs

## Step 1: TPU Setup

In [ ]:
import os
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

try:
    DEVICE = xm.xla_device()
    test_tensor = torch.zeros((2, 3), device=DEVICE)
    print('TPU device:', DEVICE)
    print('TPU runtime: OK')
except Exception as e:
    raise RuntimeError(f'TPU not accessible: {e}. '
        'Go to Notebook Settings → Accelerator → TPU → Save. '
        'Then restart (Kernel → Restart & clear output).')

## Step 2: Clone + Install

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

## Step 3: Imports + Config

In [ ]:
import sys, os, json, math, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, '/kaggle/working/ThermoRG-NN')
from thermorg.j_topo import compute_J_topo

plt.rcParams.update({'font.size': 13, 'axes.labelsize': 14, 'axes.titlesize': 14,
                     'figure.figsize': (6, 4.5), 'figure.dpi': 130})

WORK_DIR = '/kaggle/working/ThermoRG-NN/experiments/phase_b2_ranking_stability'
os.makedirs(WORK_DIR, exist_ok=True)

# ── Experiment config ─────────────────────────────────────────────────
ARCHITECTURES = [
    {'name': 'ThermoNet-6-W32-BN', 'width': 32, 'depth': 6, 'norm': 'BN', 'skip': False},
    {'name': 'ThermoNet-6-W64-BN', 'width': 64, 'depth': 6, 'norm': 'BN', 'skip': False},
    {'name': 'ThermoNet-6-W96-BN', 'width': 96, 'depth': 6, 'norm': 'BN', 'skip': False},
    {'name': 'ThermoNet-3-W64-BN', 'width': 64, 'depth': 3, 'norm': 'BN', 'skip': False},
    {'name': 'ThermoNet-9-W48-BN', 'width': 48, 'depth': 9, 'norm': 'BN', 'skip': False},
]
N_SEEDS = 5
N_EPOCHS = 50
BATCH_SIZE = 512
LR = 0.01
MOMENTUM = 0.9

print(f"Config: {len(ARCHITECTURES)} archs × {N_SEEDS} seeds × {N_EPOCHS} epochs")

## Step 4: Model + Synthetic Data

In [ ]:
class ThermoNet(nn.Module):
    """ThermoNet: Conv2d + GELU, optional BN, optional skip."""
    def __init__(self, width=64, depth=6, in_channels=3, num_classes=10,
                 norm='BN', skip=False, kernel_size=3, padding=1):
        super().__init__()
        self.width = width
        self.depth = depth
        self.norm_type = norm
        self.skip = skip
        
        layers = []
        c = in_channels
        for i in range(depth):
            layers.append(nn.Conv2d(c, width, kernel_size=kernel_size, padding=padding, bias=(norm=='None')))
            if norm == 'BN':
                layers.append(nn.BatchNorm2d(width))
            elif norm == 'LN':
                layers.append(nn.LayerNorm([width, 32, 32]))
            layers.append(nn.GELU())
            c = width
        
        if skip:
            # Residual: add skip connection every 2 layers
            pass  # skip version not needed for this experiment
        
        layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(width, num_classes)]
        self.net = nn.Sequential(*layers)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.normal_(m.weight, mean=0.0, std=1.0)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.net(x)


# ── Real CIFAR-10 data ───────────────────────────────────────────────
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

tfm_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616])
])
tfm_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616])
])
train_ds = CIFAR10(root='./data', train=True, transform=tfm_train, download=True)
test_ds  = CIFAR10(root='./data', train=False, transform=tfm_val, download=True)

N_TRAIN = len(train_ds)
N_TEST = len(test_ds)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f"CIFAR-10 train: {N_TRAIN} samples, test: {N_TEST} samples")

## Step 5: Training Loop

In [ ]:
# ── TPU-optimized training ─────────────────────────────────────────────

criterion = nn.CrossEntropyLoss()
# Use standard DataLoader on TPU
# Only xm.optimizer_step() is TPU-specific

results = {}  # {arch_name: {seed: {j_topo, final_loss}}}

total_start = time.time()

for arch in ARCHITECTURES:
    arch_name = arch['name']
    results[arch_name] = {}
    
    print(f"\n{'='*60}")
    print(f"Architecture: {arch_name} (W={arch['width']}, L={arch['depth']})")
    print(f"{'='*60}")
    
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        model = ThermoNet(
            width=arch['width'],
            depth=arch['depth'],
            norm=arch['norm'],
            skip=arch['skip'],
            num_classes=10
        ).to(DEVICE)
        optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
        
        # Compute J_topo (zero-cost, before training)
        model_cpu = copy.deepcopy(model).cpu()
        j_topo = compute_J_topo(model_cpu, layer_types=(nn.Conv2d, nn.Linear))
        del model_cpu
        
        # Train
        for epoch in range(N_EPOCHS):
            model.train()
            for Xb_tpu, yb_tpu in train_loader:
                optimizer.zero_grad()
                out = model(Xb_tpu)
                loss = criterion(out, yb_tpu)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                xm.optimizer_step(optimizer)
            scheduler.step()
        
        # Evaluate final loss
        model.eval()
        total_loss = 0
        n_batches = 0
        with torch.no_grad():
            for Xb_tpu, yb_tpu in test_loader:
                out = model(Xb_tpu)
                loss = criterion(out, yb_tpu)
                total_loss += loss.detach()
                n_batches += 1
        final_loss = (total_loss / n_batches).item()
        if np.isnan(final_loss):
            print(f'  Seed {seed}: loss=NaN, setting to 9999.0')
            final_loss = 9999.0
        
        results[arch_name][seed] = {
            'j_topo': j_topo,
            'final_loss': final_loss,
            'width': arch['width'],
            'depth': arch['depth'],
        }
        print(f"  Seed {seed}: J_topo={j_topo:.4f}, loss={final_loss:.4f}")

total_time = time.time() - total_start
print(f"\nTotal time: {total_time:.1f}s ({total_time/60:.1f} min)")

## Step 6: Analysis — Spearman Correlation

In [ ]:
# ── Aggregate and compute Spearman ρ ──────────────────────────────────────

# Per-architecture: mean loss across seeds, J_topo (should be same for all seeds since J_topo is zero-cost)
arch_summary = []
for arch_name, seed_data in results.items():
    losses = [seed_data[s]['final_loss'] for s in seed_data]
    j_topo = seed_data[0]['j_topo']  # J_topo is identical across seeds (zero-cost)
    arch_summary.append({
        'name': arch_name,
        'j_topo': j_topo,
        'mean_loss': np.mean(losses),
        'std_loss': np.std(losses),
        'width': seed_data[0]['width'],
        'depth': seed_data[0]['depth'],
    })

print("\nArchitecture Summary:")
print(f"{'Name':<25} {'J_topo':>8} {'Mean Loss':>12} {'Std Loss':>10}")
print("-" * 60)
for a in arch_summary:
    print(f"{a['name']:<25} {a['j_topo']:>8.4f} {a['mean_loss']:>12.4f} {a['std_loss']:>10.4f}")

# Spearman correlation: J_topo vs mean_loss
j_topos = np.array([a['j_topo'] for a in arch_summary])
mean_losses = np.array([a['mean_loss'] for a in arch_summary])

spearman_r, spearman_p = stats.spearmanr(j_topos, mean_losses)
print(f"\nSpearman correlation: ρ = {spearman_r:.4f}, p = {spearman_p:.4f}")
print(f"Interpretation: |ρ| = {abs(spearman_r):.3f}")
if abs(spearman_r) >= 0.7:
    print("→ J_topo ranking is STABLE across seeds (two-stage may be unnecessary)")
else:
    print("→ J_topo ranking is SEED-DEPENDENT (two-stage heuristic is justified)")

## Step 7: Visualization

In [ ]:
# ── Scatter plot: J_topo vs Mean Loss ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: scatter with error bars
ax = axes[0]
x = [a['j_topo'] for a in arch_summary]
y = [a['mean_loss'] for a in arch_summary]
yerr = [a['std_loss'] for a in arch_summary]

ax.errorbar(x, y, yerr=yerr, fmt='o', capsize=5, markersize=8,
            color='steelblue', ecolor='lightgray', elinewidth=1.5)
for i, a in enumerate(arch_summary):
    ax.annotate(a['name'].replace('ThermoNet-', 'TN-'), (x[i], y[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# Fit line
slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
x_fit = np.linspace(min(x)-0.02, max(x)+0.02, 100)
y_fit = slope * x_fit + intercept
ax.plot(x_fit, y_fit, 'r--', linewidth=1.5, label=f'ρ={spearman_r:.3f}')

ax.set_xlabel('$J_{\mathrm{topo}}$')
ax.set_ylabel('Mean Validation Loss')
ax.set_title(f'J_topo vs Loss (5 archs × 5 seeds)\nSpearman ρ = {spearman_r:.3f} (p = {spearman_p:.3f})')
ax.grid(True, alpha=0.3)
ax.legend()

# Right: bar chart of per-seed losses for each arch
ax2 = axes[1]
arch_names = [a['name'].replace('ThermoNet-', '') for a in arch_summary]
x_pos = np.arange(len(arch_names))
bar_width = 0.15

colors = plt.cm.viridis(np.linspace(0.2, 0.8, N_SEEDS))
for seed in range(N_SEEDS):
    seed_losses = [results[a['name']][seed]['final_loss'] for a in arch_summary]
    ax2.bar(x_pos + seed * bar_width, seed_losses, bar_width, 
            label=f'Seed {seed}', color=colors[seed], alpha=0.8)

ax2.set_xticks(x_pos + bar_width * 2)
ax2.set_xticklabels(arch_names, rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('Validation Loss')
ax2.set_title('Per-Seed Loss Variance')
ax2.legend(fontsize=7, ncol=3)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Phase B2: Per-Seed Ranking Stability', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'ranking_stability.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: ranking_stability.png")

## Step 8: Save Results

In [ ]:
# ── Save structured results ──────────────────────────────────────────────

output = {
    'experiment': 'phase_b2_ranking_stability',
    'description': 'Per-seed ranking stability of J_topo vs loss',
    'config': {
        'architectures': ARCHITECTURES,
        'n_seeds': N_SEEDS,
        'n_epochs': N_EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'momentum': MOMENTUM,
    },
    'spearman': {
        'rho': spearman_r,
        'p_value': spearman_p,
    },
    'arch_summary': arch_summary,
    'raw_results': results,
    'total_time_minutes': total_time / 60,
    'interpretation': (
        'J_topo ranking STABLE (|rho|>=0.7)' if abs(spearman_r) >= 0.7
        else 'J_topo ranking SEED-DEPENDENT (|rho|<0.7, two-stage heuristic justified)'
    ),
}

results_path = os.path.join(WORK_DIR, 'ranking_stability_results.json')
with open(results_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f"Results saved to: {results_path}")

## Step 9: Git Push

In [ ]:
%cd /kaggle/working/ThermoRG-NN
!git add experiments/phase_b2_ranking_stability/
!git status

In [ ]:
# Uncomment to commit and push:
# !git commit -m "Phase B2: per-seed ranking stability experiment"
# !git push origin develop